# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [6]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
parquet_files

['../../05_src/data/prices/CTAS/CTAS_2002/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2005/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2004/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2003/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2010/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2017/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2021/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2019/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2018/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2020/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2016/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2011/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2008/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2006/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2001/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2000/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2007/part.0.parquet

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [7]:
# Write your code below.
import dask.dataframe as dd
ddf = dd.read_parquet(parquet_files)
ddf.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03,NaN,43.382847,47.562963,40.596099,47.449986,4674353.0,2000
A,2000-01-04,NaN,40.068871,41.499902,39.014426,41.047995,4765083.0,2000
A,2000-01-05,NaN,37.583397,40.068874,36.340662,39.918237,5758642.0,2000
A,2000-01-06,NaN,36.152367,37.357448,35.022605,37.131495,2534434.0,2000
A,2000-01-07,NaN,39.165066,39.729947,35.549831,35.587487,2819626.0,2000


In [8]:
ddf = ddf.groupby('Ticker', group_keys=False).apply(lambda x: x.assign(Adj_Close_lag=x['Adj Close'].shift(1)))
print(ddf.columns)

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year',
       'Adj_Close_lag'],
      dtype='object', name='Price')


/var/folders/0f/kv64z9t57q36ccz9cbh3qfd80000gn/T/ipykernel_81952/902945308.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf = ddf.groupby('Ticker', group_keys=False).apply(lambda x: x.assign(Adj_Close_lag=x['Adj Close'].shift(1)))


In [9]:
# Calculate returns based on Adjusted Close
ddf['returns'] = (ddf['Adj Close'] / ddf['Adj_Close_lag']) - 1 

# Calculate the high-low range
ddf['hi_lo_range'] = ddf['High'] - ddf['Low']

# Assign the result to dd_feat
dd_feat = ddf[['Date', 'Close', 'Adj Close', 'Adj_Close_lag', 'returns',
       'hi_lo_range']]

In [14]:
dd_feat.compute()

Price,Date,Close,Adj Close,Adj_Close_lag,returns,hi_lo_range
Ticker,,,,,,
DOV,2007-01-03,22.914824,NaN,NaN,NaN,0.507130
DOV,2007-01-04,23.013441,NaN,NaN,NaN,0.394437
DOV,2007-01-05,22.727003,NaN,NaN,NaN,0.366264
DOV,2007-01-08,22.750484,NaN,NaN,NaN,0.342784
DOV,2007-01-09,22.863167,NaN,NaN,NaN,0.230087
...,...,...,...,...,...,...
CTLT,2012-12-24,NaN,NaN,NaN,NaN,NaN
CTLT,2012-12-26,NaN,NaN,NaN,NaN,NaN
CTLT,2012-12-27,NaN,NaN,NaN,NaN,NaN


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [12]:
import pandas as pd
import numpy as np

In [15]:
# Convert the Dask DataFrame to a Pandas DataFrame
pandas_df = dd_feat.compute()

# Add a rolling average return calculation with a window of 10 days
pandas_df['rolling_avg_return'] = pandas_df['returns'].rolling(window=10).mean()

# Optionally, display the result
pandas_df.head(15)

Price,Date,Close,Adj Close,Adj_Close_lag,returns,hi_lo_range,rolling_avg_return
Ticker,,,,,,,
DOV,2006-01-03,19.198721,NaN,NaN,NaN,0.578414,NaN
DOV,2006-01-04,19.527254,NaN,NaN,NaN,0.458103,NaN
DOV,2006-01-05,19.430079,NaN,NaN,NaN,0.374811,NaN
DOV,2006-01-06,19.753998,NaN,NaN,NaN,0.231366,NaN
DOV,2006-01-09,20.031631,NaN,NaN,NaN,0.384066,NaN
DOV,2006-01-10,20.040880,NaN,NaN,NaN,0.310030,NaN
DOV,2006-01-11,20.175079,NaN,NaN,NaN,0.379439,NaN
DOV,2006-01-12,19.976110,NaN,NaN,NaN,0.541395,NaN
DOV,2006-01-13,19.989985,NaN,NaN,NaN,0.208229,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

> No

> depends on the dataset size. If we have a smaller dataset, pandas work faster. Dask is better for larger datasets as it processes data in chunks and supports parallel processing, making it more efficient when dealing with data that doesn’t fit into memory.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ x ] Created a branch with the correct naming convention.
- [ x ] Ensured that the repository is public.
- [ x ] Reviewed the PR description guidelines and adhered to them.
- [ x ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.